# ⚖️ RAG vs. Vanilla LLM

**Day 7 — Notebook 4 of 4 | Estimated Time: 20 minutes**

---

## What You'll Learn
- Run direct side-by-side comparisons of RAG vs vanilla LLM responses
- Identify the specific failure modes RAG fixes (and doesn't fix)
- Use LLM-as-judge to score response quality automatically
- Understand when RAG is overkill and when it's essential

---

## Setup

In [ ]:
%pip install -U -q "google-genai>=1.0.0" chromadb langchain-text-splitters

In [ ]:
import sys, os, json, hashlib
import chromadb
from langchain_text_splitters import RecursiveCharacterTextSplitter

repo_root = os.path.abspath(os.path.join(os.getcwd(), "../../.."))
if repo_root not in sys.path:
    sys.path.append(repo_root)

from helpers.auth import get_api_key
from google import genai
from google.genai import types
from IPython.display import Markdown

client = genai.Client(api_key=get_api_key())
EMBEDDING_MODEL = "text-embedding-004"
GEN_MODEL = "gemini-2.5-flash"
print("✅ Ready!")

---

## 1. Build the RAG System

We'll use a **private company policy document** that the LLM was definitely not trained on.

In [ ]:
# Private company policy — fictional and specific enough that the LLM cannot know it
NOVA_TECH_POLICIES = [
    {
        "title": "Leave Policy",
        "text": "Nova Tech employees receive 22 days annual leave per year. Leave accrues at 1.83 days "
                "per month. Unused leave above 3 days cannot be carried over. Leave requests must be "
                "submitted via LeaveBot at least 5 working days in advance for absences of 2+ days. "
                "Emergency leave up to 2 days can be approved same-day by a line manager.",
    },
    {
        "title": "Remote Work",
        "text": "Nova Tech allows a maximum of 4 remote days per week. All employees must work from "
                "the office on Tuesdays (Team Sync day). The home office allowance is $800 per year. "
                "International remote work requires a Temporary Remote Work Agreement (TRWA) filed "
                "with HR at least 14 days before travel. Maximum continuous international remote "
                "period is 30 days per calendar year.",
    },
    {
        "title": "Compensation and Equity",
        "text": "Nova Tech benchmarks salaries at the 65th percentile of the relevant market. "
                "Compensation reviews happen every July. Equity grants vest over 4 years with a "
                "12-month cliff. All permanent employees at L3+ receive an annual equity refresh "
                "of 0.05% of their original grant. The employee stock purchase plan (ESPP) allows "
                "purchases at a 15% discount with a 6-month offering period.",
    },
    {
        "title": "Learning Budget",
        "text": "The Nova Tech learning budget is $2,000 per employee per year. It covers online "
                "courses, books, conferences, and certifications. Up to $500 may be used for "
                "conference attendance (travel and accommodation included). Requests above $300 "
                "require manager approval via the LMS portal. Budget resets every January 1st "
                "and cannot be carried over.",
    },
    {
        "title": "Parental Leave",
        "text": "Nova Tech provides 20 weeks fully paid leave for primary caregivers and 6 weeks "
                "for secondary caregivers. Employees must have been with the company for at least "
                "6 months to qualify. A gradual return-to-work program allows 50% hours for "
                "the first 4 weeks after returning. Adoption and foster placement leave follows "
                "the same structure as birth parental leave.",
    },
]

# Quick RAG build
chroma = chromadb.Client()
col = chroma.create_collection("nova_tech", metadata={"hnsw:space": "cosine"})

texts = [p["text"] for p in NOVA_TECH_POLICIES]
result = client.models.embed_content(
    model=EMBEDDING_MODEL,
    contents=texts,
    config=types.EmbedContentConfig(task_type="RETRIEVAL_DOCUMENT"),
)
col.add(
    ids=[str(i) for i in range(len(NOVA_TECH_POLICIES))],
    documents=texts,
    embeddings=[e.values for e in result.embeddings],
    metadatas=[{"title": p["title"]} for p in NOVA_TECH_POLICIES],
)
print(f"✅ Indexed {col.count()} policy chunks")

In [ ]:
def vanilla_llm(question: str) -> str:
    """Ask the LLM directly, with no context."""
    response = client.models.generate_content(
        model=GEN_MODEL,
        contents=question,
        config=types.GenerateContentConfig(temperature=0.1),
    )
    return response.text.strip()


def rag_answer(question: str, top_k: int = 2) -> tuple[str, list[str]]:
    """Retrieve context and generate a grounded answer."""
    q_vec = client.models.embed_content(
        model=EMBEDDING_MODEL,
        contents=question,
        config=types.EmbedContentConfig(task_type="RETRIEVAL_QUERY"),
    ).embeddings[0].values

    results = col.query(query_embeddings=[q_vec], n_results=top_k,
                        include=["documents", "metadatas"])
    chunks = results["documents"][0]
    sources = [m["title"] for m in results["metadatas"][0]]
    context = "\n\n".join(f"[{s}] {c}" for s, c in zip(sources, chunks))

    response = client.models.generate_content(
        model=GEN_MODEL,
        contents=f"""Answer using ONLY the Nova Tech policy context below.
If the answer is not in the context, say so explicitly.

<context>
{context}
</context>

Question: {question}""",
        config=types.GenerateContentConfig(temperature=0.1),
    )
    return response.text.strip(), sources

---

## 2. Side-by-Side Comparison

In [ ]:
comparison_questions = [
    {
        "question": "How many days of annual leave do Nova Tech employees get?",
        "ground_truth": "22 days",
        "test_type": "Private/specific fact",
    },
    {
        "question": "What is the maximum number of remote days allowed per week at Nova Tech?",
        "ground_truth": "4 days (must be in office on Tuesdays)",
        "test_type": "Company-specific policy",
    },
    {
        "question": "How long is parental leave for a secondary caregiver at Nova Tech?",
        "ground_truth": "6 weeks fully paid",
        "test_type": "Specific number not in training data",
    },
    {
        "question": "What percentage discount does Nova Tech's ESPP offer?",
        "ground_truth": "15% discount",
        "test_type": "Niche internal detail",
    },
]

print("SIDE-BY-SIDE COMPARISON: Vanilla LLM vs RAG\n")
print("=" * 70)

for item in comparison_questions:
    q = item["question"]
    gt = item["ground_truth"]
    
    vanilla = vanilla_llm(q)
    rag, sources = rag_answer(q)
    
    print(f"\n📋 Test type: {item['test_type']}")
    print(f"❓ Question: {q}")
    print(f"✅ Ground truth: {gt}")
    print(f"\n🤖 VANILLA LLM:")
    print(f"   {vanilla[:200]}")
    print(f"\n📚 RAG (sources: {sources}):")
    print(f"   {rag[:200]}")
    print("-" * 70)

---

## 3. Where RAG Doesn't Help

In [ ]:
# Cases where RAG doesn't add value — or can hurt
no_rag_needed = [
    {
        "question": "What is the capital of France?",
        "why": "General knowledge — well in training data",
    },
    {
        "question": "Write a Python function to reverse a string.",
        "why": "Code generation — no factual retrieval needed",
    },
    {
        "question": "Translate 'hello world' into French and Spanish.",
        "why": "Translation — language capability, not factual",
    },
]

print("CASES WHERE RAG IS UNNECESSARY:\n")
for item in no_rag_needed:
    vanilla = vanilla_llm(item["question"])
    rag, _ = rag_answer(item["question"])

    print(f"❓ {item['question']}")
    print(f"   Why RAG isn't needed: {item['why']}")
    print(f"   Vanilla: {vanilla[:120]}")
    print(f"   RAG:     {rag[:120]}")
    print()

---

## 4. LLM-as-Judge Evaluation

Use Gemini itself to automatically score the quality of both approaches.

In [ ]:
from pydantic import BaseModel

class EvalResult(BaseModel):
    score: int          # 1–5
    is_correct: bool    # Does it match the ground truth?
    hallucination: bool # Did it fabricate specific facts?
    reasoning: str      # One-sentence explanation


def judge_answer(question: str, answer: str, ground_truth: str) -> EvalResult:
    """Use Gemini to evaluate an answer against the ground truth."""
    response = client.models.generate_content(
        model=GEN_MODEL,
        contents=f"""Evaluate the following answer to a question about company HR policy.

Question: {question}
Ground truth (correct answer): {ground_truth}
Answer to evaluate: {answer}

Score from 1-5 where:
5 = Correct and complete
4 = Mostly correct with minor gaps
3 = Partially correct
2 = Mostly wrong or vague
1 = Wrong or refuses to answer""",
        config=types.GenerateContentConfig(
            response_mime_type="application/json",
            response_schema=EvalResult,
            temperature=0.0,
        ),
    )
    return EvalResult.model_validate_json(response.text)


# Run evaluation
print("LLM-as-JUDGE EVALUATION\n")
print(f"{'Question':<45} {'Vanilla':>10} {'RAG':>8}")
print("-" * 70)

vanilla_total = 0
rag_total = 0

for item in comparison_questions:
    q = item["question"]
    gt = item["ground_truth"]

    vanilla_ans = vanilla_llm(q)
    rag_ans, _ = rag_answer(q)

    v_eval = judge_answer(q, vanilla_ans, gt)
    r_eval = judge_answer(q, rag_ans, gt)

    vanilla_total += v_eval.score
    rag_total += r_eval.score

    halluc_flag = "🚨" if v_eval.hallucination else "  "
    print(f"{q[:44]:<45} {halluc_flag}{v_eval.score}/5     {r_eval.score}/5")
    if v_eval.hallucination:
        print(f"   ⚠️  Vanilla hallucination: {v_eval.reasoning}")

avg_v = vanilla_total / len(comparison_questions)
avg_r = rag_total / len(comparison_questions)
print(f"\n{'AVERAGE':<45} {avg_v:.1f}/5   {avg_r:.1f}/5")
print(f"\nRAG improvement: +{avg_r - avg_v:.1f} points average")

---

## 5. The Decision Framework

Use this to decide whether to use RAG for a given task:

In [ ]:
def should_use_rag(task_description: str) -> str:
    """Ask Gemini to recommend RAG vs vanilla LLM for a given task."""
    response = client.models.generate_content(
        model=GEN_MODEL,
        contents=f"""For the following AI task, recommend whether to use:
A) Vanilla LLM (no retrieval)
B) RAG (retrieval augmented generation)
C) Fine-tuning

Task: {task_description}

Answer with just the letter (A, B, or C) and one sentence of reasoning.""",
        config=types.GenerateContentConfig(temperature=0.0),
    )
    return response.text.strip()


tasks = [
    "Answer questions about a company's internal HR policies",
    "Summarise a user-provided document in 3 bullet points",
    "Generate product descriptions in a specific brand voice",
    "Answer general knowledge trivia questions",
    "Answer questions about the latest news articles from today",
    "Classify customer support tickets into predefined categories",
]

print("RAG vs Vanilla vs Fine-Tune Recommendations:\n")
for task in tasks:
    rec = should_use_rag(task)
    print(f"  📌 {task}")
    print(f"     → {rec}")
    print()

---

## 🏋️ Exercise: Build Your Own Comparison

Create a knowledge base from a topic where the LLM is likely to hallucinate, then run a comparison.

In [ ]:
# Exercise: Your own RAG vs Vanilla comparison
# TODO:
# Pick a domain where the LLM doesn't have reliable knowledge:
#   - A niche hobby (obscure board game rules, regional cuisine)
#   - A fictional company's product specs
#   - Your local area's specific services or policies
#   - A technical specification from a recent standard
#
# 1. Write 3-5 "documents" with specific facts
# 2. Build a mini RAG system (reuse the pattern from this notebook)
# 3. Create 5 Q&A pairs with known correct answers
# 4. Run both vanilla LLM and RAG
# 5. Use judge_answer() to score both approaches
# 6. Write a 2-sentence conclusion: when did RAG help most?

my_documents = [
    # {"title": "...", "text": "..."},
]

my_questions = [
    # {"question": "...", "ground_truth": "..."},
]

# TODO: Build, run, and evaluate

---

## Summary: When Does RAG Help?

| Scenario | Vanilla LLM | RAG | Winner |
|----------|-------------|-----|--------|
| Private company policies | ❌ Guesses or refuses | ✅ Accurate | RAG |
| Recent events / news | ❌ Knowledge cutoff | ✅ (with live data) | RAG |
| General knowledge (capitals, history) | ✅ Usually correct | ✅ No change | Vanilla |
| Creative writing | ✅ No retrieval needed | ⚠️ May constrain creativity | Vanilla |
| Code generation | ✅ Trained on code | ⚠️ Unnecessary overhead | Vanilla |
| Domain-specific jargon/facts | ❌ May hallucinate | ✅ Grounded | RAG |
| Long document Q&A | ❌ Can't process full doc | ✅ Retrieves relevant chunks | RAG |

---

## Day 7 Complete! 🎉

You've learned:
- ✅ RAG architecture and when to use it (Notebook 1)
- ✅ Building a complete RAG system from scratch with Gemini + ChromaDB (Notebook 2)
- ✅ Streamlined RAG chains with LangChain LCEL (Notebook 3)
- ✅ Side-by-side comparison and LLM-as-judge evaluation (Notebook 4)

**Next:** Day 8 — Advanced RAG Techniques

---

## 📚 Further Reading

| Resource | Type | Link |
|----------|------|------|
| RAG vs Fine-Tuning | Blog | [aws.amazon.com/blogs/machine-learning/rag-vs-fine-tuning](https://aws.amazon.com/blogs/machine-learning/rag-vs-fine-tuning/) |
| Evaluating RAG | Blog | [docs.ragas.io](https://docs.ragas.io/) |